In [1]:
# Import librearies to talk with Spark and functions
from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name, regexp_extract, regexp_replace, split, element_at

In [2]:
# Create the session
spark = SparkSession.builder \
    .appName("CA2") \
    .enableHiveSupport() \
    .getOrCreate()

25/11/28 17:08:06 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


#### Stock prices stored in Hadoop will be retrieved and subsequently persisted in Hadoop/Hive using Spark and Spark SQL

In [3]:
# Define a constanst with the input path and read the csv file from Hadoop
filePathInputStockPrice = "/CA2/input/stockprice"

df = spark.read.csv(filePathInputStockPrice, header = True, inferSchema = True)

In [4]:
# I split the file path in several parts using / as a seperator
# Then, there is a file name (company name) which has ^, it is removed
# lastly we get the names of the files to create a new field named company
# which has the name of the company (filename)
df = df.withColumn(
    "filename",
    element_at(split(input_file_name(), "/"), -1)
).withColumn(
    "clean_filename",
    regexp_replace("filename", "%5E", "")   
).withColumn(
    "company",
    regexp_extract("clean_filename", r"([A-Za-z0-9]+)\.csv$", 1)
).drop("filename", "clean_filename")

In [5]:
# I select the company field 
df.select("company").distinct().show(999, truncate=False)

+-------+
|company|
+-------+
|XOM    |
|CCL    |
|DIS    |
|TSLA   |
|PG     |
|B      |
|PFE    |
|WMT    |
|LOW    |
|BAC    |
|MSFT   |
|AAPL   |
|JPM    |
|MCD    |
|V      |
|GOOG   |
|JNJ    |
|HD     |
|PYPL   |
|AMZN   |
|UPS    |
|KO     |
|TSM    |
|META   |
|AMT    |
|FB     |
|CVX    |
|GSPC   |
|GOOGL  |
|BA     |
|TM     |
|NKE    |
|BABA   |
|NFLX   |
|UNH    |
|ABNB   |
|A      |
|SBUX   |
|MA     |
|BKNG   |
|NVDA   |
+-------+



In [6]:
# I count how many companies are in the dataframe
df.select("company").distinct().count()

41

In [7]:
# show the table without the company name
df.select([c for c in df.columns if c != "company"]).show(999, truncate=False)

+----------+------------------+------------------+------------------+------------------+------------------+---------+
|Date      |Open              |High              |Low               |Close             |Adj Close         |Volume   |
+----------+------------------+------------------+------------------+------------------+------------------+---------+
|2019-12-31|36.80265808105469 |37.191650390625   |36.72675323486328 |37.17267608642578 |31.371051788330078|15175703 |
|2020-01-02|37.286529541015625|37.33396530151367 |36.88804626464844 |37.13472366333008 |31.339025497436523|16514072 |
|2020-01-03|36.736244201660156|37.2296028137207  |36.688804626464844|36.93548583984375 |31.170879364013672|14922848 |
|2020-01-06|36.831119537353516|37.00189971923828 |36.71726989746094 |36.88804626464844 |31.13084602355957 |15771951 |
|2020-01-07|37.11574935913086 |37.12523651123047 |36.69829177856445 |36.764705657958984|31.026758193969727|20108107 |
|2020-01-08|36.774192810058594|37.21062469482422 |36.764

In [8]:
# count how many records are in the dataframe
df.count()

10175

In [9]:
# count how many rows per company
df.groupBy("company").count().show(999, truncate=False)

+-------+-----+
|company|count|
+-------+-----+
|XOM    |254  |
|CCL    |254  |
|DIS    |254  |
|TSLA   |254  |
|PG     |254  |
|B      |254  |
|PFE    |254  |
|WMT    |254  |
|LOW    |254  |
|BAC    |254  |
|MSFT   |254  |
|AAPL   |254  |
|JPM    |254  |
|MCD    |254  |
|V      |254  |
|GOOG   |254  |
|JNJ    |254  |
|HD     |254  |
|PYPL   |254  |
|AMZN   |254  |
|UPS    |254  |
|KO     |254  |
|TSM    |254  |
|META   |254  |
|AMT    |254  |
|FB     |254  |
|CVX    |254  |
|GSPC   |254  |
|GOOGL  |254  |
|BA     |254  |
|TM     |254  |
|NKE    |254  |
|BABA   |254  |
|NFLX   |254  |
|UNH    |254  |
|ABNB   |15   |
|A      |254  |
|SBUX   |254  |
|MA     |254  |
|BKNG   |254  |
|NVDA   |254  |
+-------+-----+



In [11]:
# Read tables from Hive Metastore before creating the table stock_prices
spark.sql("SHOW TABLES").show()

+---------+----------+-----------+
|namespace| tableName|isTemporary|
+---------+----------+-----------+
|  default| test_hive|      false|
|  default|test_hive2|      false|
+---------+----------+-----------+



#### Store the data in Hadoop using Spark SQL

In [12]:
#Create table in Hive with its fields and type
# and the field for which it is partitioned
spark.sql("""
    CREATE TABLE IF NOT EXISTS stock_prices (
        Date DATE,
        Open DOUBLE,
        High DOUBLE,
        Low DOUBLE,
        Close DOUBLE,
        AdjClose DOUBLE,
        Volume BIGINT
    )
    PARTITIONED BY (company STRING)
    STORED AS PARQUET
""")


25/11/28 17:08:54 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


DataFrame[]

In [13]:
# Read tables from Hive Metastore after creating the table stock_prices
spark.sql("SHOW TABLES").show()

+---------+------------+-----------+
|namespace|   tableName|isTemporary|
+---------+------------+-----------+
|  default|stock_prices|      false|
|  default|   test_hive|      false|
|  default|  test_hive2|      false|
+---------+------------+-----------+



In [14]:
# Transform the name of the column in the data frame to match with the name of the table created in Hive
df = df.withColumnRenamed("Adj Close", "AdjClose")

In [15]:
# Show the dataframe 
df.select([c for c in df.columns if c != "company"]).show()

+----------+------------------+------------------+------------------+------------------+------------------+--------+
|      Date|              Open|              High|               Low|             Close|          AdjClose|  Volume|
+----------+------------------+------------------+------------------+------------------+------------------+--------+
|2019-12-31| 36.80265808105469|   37.191650390625| 36.72675323486328| 37.17267608642578|31.371051788330078|15175703|
|2020-01-02|37.286529541015625| 37.33396530151367| 36.88804626464844| 37.13472366333008|31.339025497436523|16514072|
|2020-01-03|36.736244201660156|  37.2296028137207|36.688804626464844| 36.93548583984375|31.170879364013672|14922848|
|2020-01-06|36.831119537353516| 37.00189971923828| 36.71726989746094| 36.88804626464844| 31.13084602355957|15771951|
|2020-01-07| 37.11574935913086| 37.12523651123047| 36.69829177856445|36.764705657958984|31.026758193969727|20108107|
|2020-01-08|36.774192810058594| 37.21062469482422|36.76470565795

In [16]:
# I append the data in Hive using the table stock_prices previously created
df.write.mode("append").insertInto("stock_prices")

25/11/28 17:09:06 WARN MemoryManager: Total allocation exceeds 50.00% (477,364,224 bytes) of heap memory
Scaling row group sizes to 88.92% for 4 writers
25/11/28 17:09:07 WARN MemoryManager: Total allocation exceeds 50.00% (477,364,224 bytes) of heap memory
Scaling row group sizes to 88.92% for 4 writers
25/11/28 17:09:08 WARN MemoryManager: Total allocation exceeds 50.00% (477,364,224 bytes) of heap memory
Scaling row group sizes to 88.92% for 4 writers
25/11/28 17:09:08 WARN MemoryManager: Total allocation exceeds 50.00% (477,364,224 bytes) of heap memory
Scaling row group sizes to 88.92% for 4 writers
25/11/28 17:09:08 WARN MemoryManager: Total allocation exceeds 50.00% (477,364,224 bytes) of heap memory
Scaling row group sizes to 88.92% for 4 writers
25/11/28 17:09:08 WARN MemoryManager: Total allocation exceeds 50.00% (477,364,224 bytes) of heap memory
Scaling row group sizes to 88.92% for 4 writers
25/11/28 17:09:08 WARN MemoryManager: Total allocation exceeds 50.00% (477,364,224

#### Performs Operations using Spark SQL

In [17]:
# Show the partitiona created by ticker/company
spark.sql("SHOW PARTITIONS stock_prices").show(999, truncate=False)

+-------------+
|partition    |
+-------------+
|company=A    |
|company=AAPL |
|company=ABNB |
|company=AMT  |
|company=AMZN |
|company=B    |
|company=BA   |
|company=BABA |
|company=BAC  |
|company=BKNG |
|company=CCL  |
|company=CVX  |
|company=DIS  |
|company=FB   |
|company=GOOG |
|company=GOOGL|
|company=GSPC |
|company=HD   |
|company=JNJ  |
|company=JPM  |
|company=KO   |
|company=LOW  |
|company=MA   |
|company=MCD  |
|company=META |
|company=MSFT |
|company=NFLX |
|company=NKE  |
|company=NVDA |
|company=PFE  |
|company=PG   |
|company=PYPL |
|company=SBUX |
|company=TM   |
|company=TSLA |
|company=TSM  |
|company=UNH  |
|company=UPS  |
|company=V    |
|company=WMT  |
|company=XOM  |
+-------------+



In [18]:
# Check if there are null values
spark.sql("SELECT * FROM stock_prices WHERE Date IS NULL OR Open IS NULL OR High IS NULL OR Low IS NULL OR Close IS NULL OR AdjClose IS NULL OR Volume IS NULL").show()

+----+----+----+---+-----+--------+------+-------+
|Date|Open|High|Low|Close|AdjClose|Volume|company|
+----+----+----+---+-----+--------+------+-------+
+----+----+----+---+-----+--------+------+-------+



In [19]:
# Select the company field and count how many rows for that company 
spark.sql("SELECT company, COUNT(*) FROM stock_prices GROUP BY company").show(999, truncate=False)

+-------+--------+
|company|count(1)|
+-------+--------+
|AAPL   |254     |
|META   |254     |
|TSLA   |254     |
|GOOG   |254     |
|FB     |254     |
|GSPC   |254     |
|GOOGL  |254     |
|BA     |254     |
|PYPL   |254     |
|AMZN   |254     |
|NVDA   |254     |
|TSM    |254     |
|XOM    |254     |
|JPM    |254     |
|DIS    |254     |
|NFLX   |254     |
|CVX    |254     |
|LOW    |254     |
|HD     |254     |
|SBUX   |254     |
|MSFT   |254     |
|BABA   |254     |
|CCL    |254     |
|B      |254     |
|MCD    |254     |
|V      |254     |
|PFE    |254     |
|UNH    |254     |
|WMT    |254     |
|NKE    |254     |
|MA     |254     |
|BKNG   |254     |
|UPS    |254     |
|PG     |254     |
|AMT    |254     |
|JNJ    |254     |
|ABNB   |15      |
|A      |254     |
|BAC    |254     |
|TM     |254     |
|KO     |254     |
+-------+--------+



In [20]:
# Select data, company and close fields for the company/ticket AAPL, order by date
spark.sql("SELECT Date, Company, CLose FROM stock_prices WHERE company = 'AAPL' ORDER BY Date").show(999, truncate=False)

+----------+-------+------------------+
|Date      |Company|CLose             |
+----------+-------+------------------+
|2019-12-31|AAPL   |73.4124984741211  |
|2020-01-02|AAPL   |75.0875015258789  |
|2020-01-03|AAPL   |74.35749816894531 |
|2020-01-06|AAPL   |74.94999694824219 |
|2020-01-07|AAPL   |74.59750366210938 |
|2020-01-08|AAPL   |75.79750061035156 |
|2020-01-09|AAPL   |77.40750122070312 |
|2020-01-10|AAPL   |77.5824966430664  |
|2020-01-13|AAPL   |79.23999786376953 |
|2020-01-14|AAPL   |78.16999816894531 |
|2020-01-15|AAPL   |77.83499908447266 |
|2020-01-16|AAPL   |78.80999755859375 |
|2020-01-17|AAPL   |79.68250274658203 |
|2020-01-21|AAPL   |79.14250183105469 |
|2020-01-22|AAPL   |79.42500305175781 |
|2020-01-23|AAPL   |79.80750274658203 |
|2020-01-24|AAPL   |79.57749938964844 |
|2020-01-27|AAPL   |77.23750305175781 |
|2020-01-28|AAPL   |79.42250061035156 |
|2020-01-29|AAPL   |81.08499908447266 |
|2020-01-30|AAPL   |80.96749877929688 |
|2020-01-31|AAPL   |77.37750244140625 |


In [21]:
# Select company, calculate the min of low  and max of high for that company 
spark.sql("SELECT company, MIN(Low) AS min_price, MAX(High) AS max_price FROM stock_prices GROUP BY company").show(999, truncate=False)

+-------+------------------+------------------+
|company|min_price         |max_price         |
+-------+------------------+------------------+
|AAPL   |53.15250015258789 |138.7899932861328 |
|META   |137.10000610351562|304.6700134277344 |
|TSLA   |23.367332458496094|239.57333374023438|
|GOOG   |50.67679977416992 |92.36000061035156 |
|FB     |137.10000610351562|304.6700134277344 |
|GSPC   |2191.860107421875 |3760.199951171875 |
|GOOGL  |50.44350051879883 |92.19149780273438 |
|BA     |89.0              |349.95001220703125|
|PYPL   |82.06999969482422 |244.25            |
|AMZN   |81.30149841308594 |177.6125030517578 |
|NVDA   |45.16999816894531 |147.2675018310547 |
|TSM    |42.70000076293945 |109.69999694824219|
|XOM    |30.110000610351562|71.37000274658203 |
|JPM    |76.91000366210938 |141.10000610351562|
|DIS    |79.06999969482422 |183.39999389648438|
|NFLX   |290.25            |575.3699951171875 |
|CVX    |51.599998474121094|122.72000122070312|
|LOW    |60.0              |180.66999816

In [22]:
# Average closing price per company
spark.sql("""
SELECT 
    company,
    AVG(close) AS avg_close_price
FROM stock_prices
GROUP BY company
ORDER BY avg_close_price DESC
""").show()

+-------+------------------+
|company|   avg_close_price|
+-------+------------------+
|      A|307763.65748031496|
|   GSPC| 3217.906731552965|
|   BKNG|1742.6955100232221|
|   NFLX| 446.3419682059701|
|     MA|  309.422875832385|
|    UNH| 301.2553945826733|
|     HD|248.73433084563007|
|    AMT|241.53204694132165|
|   BABA|239.77440973717395|
|   META| 234.4355120921698|
|     FB| 234.4355120921698|
|      B|205.09484280563714|
|    MCD|200.87220511849472|
|     BA| 197.6016537523645|
|      V|193.92822830320344|
|   MSFT|192.88704717440868|
|   PYPL|161.31466524619756|
|   ABNB| 147.6260009765625|
|    JNJ|145.81661416601946|
|    LOW|135.19763787337175|
+-------+------------------+
only showing top 20 rows



In [23]:
# Estimate daily difference, which measures how much the stock changed from open to close 
spark.sql("""
SELECT 
    company,
    STDDEV((Close - Open) / Open) AS daily_diff
FROM stock_prices
GROUP BY company
ORDER BY daily_diff DESC
""").show()

+-------+--------------------+
|company|          daily_diff|
+-------+--------------------+
|    CCL| 0.05070670941759238|
|   ABNB|  0.0429063601747753|
|   TSLA|0.040588852013142725|
|     BA|0.033354358310102264|
|   NVDA|  0.0266689002153799|
|    CVX|0.024559655720012827|
|    XOM|0.023702001746838307|
|   NFLX|0.023367299260864408|
|    BAC|0.022859933198362964|
|   PYPL| 0.02281996112134665|
|    AMT| 0.02197104840519689|
|    LOW|0.021873835324502787|
|    UNH| 0.02182274067080241|
|   BKNG|0.020783212545614216|
|    DIS|0.020100844067323872|
|   AAPL|   0.020061367343923|
|    JPM|0.019957817753106896|
|     MA|0.019757366583819345|
|    UPS|0.019339068797135206|
|   META|0.019330568382181543|
+-------+--------------------+
only showing top 20 rows



#### Tweets stored in Hadoop will be retrieve and subsequently persisted in Mongo using pymongo

In [23]:
filePathInputStockTweet = "/CA2/input/stocktweet"

# Every option represents: column names, detect column types, Some characters cause the original CSV file to insert a line break inside the tweet text,
# Double quotes define the boundary of text fields, Commas inside tweets break the row into wrong columns.
# Allows quotes inside quoted text, When Spark reads a CSV field, it removes spaces at the beginning of that field
# Spark removes spaces at the end of a field

dfStockTweet = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .option("ignoreLeadingWhiteSpace", True)
        .option("ignoreTrailingWhiteSpace", True)
        .csv(filePathInputStockTweet)
)

In [24]:
dfStockTweet.show(15)

+------+----------+------+--------------------+
|    id|      date|ticker|               tweet|
+------+----------+------+--------------------+
|100001|01/01/2020|  AMZN|$AMZN Dow futures...|
|100002|01/01/2020|  TSLA|$TSLA Daddy's dri...|
|100003|01/01/2020|  AAPL|$AAPL We’ll been ...|
|100004|01/01/2020|  TSLA|$TSLA happy new y...|
|100005|01/01/2020|  TSLA|$TSLA haha just a...|
|100006|01/01/2020|  TSLA|$TSLA NOBODY: Gas...|
|100007|02/01/2020|  AAPL|$AAPL $300 calls ...|
|100008|02/01/2020|  AAPL|$AAPL Remember, i...|
|100009|02/01/2020|  AAPL|$AAPL called it, ...|
|100010|02/01/2020|    HD|$HD Bought more a...|
|100011|02/01/2020|  AAPL|Apple is taking t...|
|100012|02/01/2020|  AAPL|$AAPL not a bad d...|
|100013|02/01/2020|  AAPL|$AAPL where are a...|
|100014|03/01/2020|  NVDA|$NVDA This should...|
|100015|03/01/2020|  AAPL|$AAPL tomorrow bu...|
+------+----------+------+--------------------+
only showing top 15 rows



In [25]:
# check if there is no null values for the three first fields which could be filled with
# incorrect characthers if we don't read the csv file properly
dfStockTweet.filter("ticker IS NULL OR date IS NULL OR id is NULL").show(truncate=False)

+---+----+------+-----+
|id |date|ticker|tweet|
+---+----+------+-----+
+---+----+------+-----+



In [26]:
from pymongo import MongoClient

# MongoDB connection details
username = "bdsp1"
password = "bdsp1"
host = "127.0.0.1"
port = 27017
database_name = "bdsp1"
collection_name = "stocktweet"

# Correct authSource
client = MongoClient(f"mongodb://{username}:{password}@{host}:{port}/?authSource={database_name}")

# Access the database
db = client[database_name]

# Create or access the collection
collection = db[collection_name]


In [27]:
batch = []
batch_size = 1000

# Insert records into the collection
try:

    for row in dfStockTweet.toLocalIterator():
        batch.append(row.asDict())

        if len(batch) >= batch_size:
            collection.insert_many(batch)
            batch = []

    # insert leftover
    if batch:
        collection.insert_many(batch)
        
except Exception as e:
    print(f"An error occurred: {e}")

#### Perform some operations

In [28]:
# Retrieve only one document
doc = collection.find_one()
print(doc)

{'_id': ObjectId('692995104988782702a901e6'), 'id': 100001, 'date': '01/01/2020', 'ticker': 'AMZN', 'tweet': '$AMZN Dow futures up by 100 points already 🥳'}


In [29]:
# Filter documents (like SQL WHERE) by ticker = AAPL
records = collection.find({"ticker": "AAPL"})
for doc in records:
    print(doc)

{'_id': ObjectId('692995104988782702a901e8'), 'id': 100003, 'date': '01/01/2020', 'ticker': 'AAPL', 'tweet': '$AAPL We’ll been riding since last December from $172.12 what to do. Decisions decisions hmm 🤔. I have 20 mins to decide. Any suggestions?'}
{'_id': ObjectId('692995104988782702a901ec'), 'id': 100007, 'date': '02/01/2020', 'ticker': 'AAPL', 'tweet': '$AAPL $300 calls First trade of 2020 Congrats to all bulls 😈'}
{'_id': ObjectId('692995104988782702a901ed'), 'id': 100008, 'date': '02/01/2020', 'ticker': 'AAPL', 'tweet': '$AAPL Remember, if you short every day, one of those days you will be right 😏'}
{'_id': ObjectId('692995104988782702a901ee'), 'id': 100009, 'date': '02/01/2020', 'ticker': 'AAPL', 'tweet': '$AAPL called it, the bear comment below makes me chuckle inside. So sweeet 😜'}
{'_id': ObjectId('692995104988782702a901f0'), 'id': 100011, 'date': '02/01/2020', 'ticker': 'AAPL', 'tweet': 'Apple is taking things UP in 2020 🚀🚀 $AAPL'}
{'_id': ObjectId('692995104988782702a901f1

In [30]:
# Counts tweets per ticker

query = [
    {"$group": {"_id": "$ticker", "tweets": {"$sum": 1}}},
    {"$sort": {"tweets": -1}}
]

records = collection.aggregate(query)

for doc in records:
    print(doc)


{'_id': 'TSLA', 'tweets': 4341}
{'_id': 'AAPL', 'tweets': 1721}
{'_id': 'BA', 'tweets': 919}
{'_id': 'DIS', 'tweets': 432}
{'_id': 'AMZN', 'tweets': 407}
{'_id': 'MSFT', 'tweets': 271}
{'_id': 'CCL', 'tweets': 264}
{'_id': 'BABA', 'tweets': 239}
{'_id': 'FB', 'tweets': 204}
{'_id': 'NFLX', 'tweets': 195}
{'_id': 'PFE', 'tweets': 186}
{'_id': 'NVDA', 'tweets': 119}
{'_id': 'WMT', 'tweets': 118}
{'_id': 'BAC', 'tweets': 65}
{'_id': 'ABNB', 'tweets': 60}
{'_id': 'XOM', 'tweets': 46}
{'_id': 'V', 'tweets': 45}
{'_id': 'SBUX', 'tweets': 45}
{'_id': 'HD', 'tweets': 39}
{'_id': 'PYPL', 'tweets': 38}
{'_id': 'JPM', 'tweets': 34}
{'_id': 'NKE', 'tweets': 29}
{'_id': 'MCD', 'tweets': 23}
{'_id': 'JNJ', 'tweets': 22}
{'_id': 'GOOG', 'tweets': 18}
{'_id': 'GOOGL', 'tweets': 17}
{'_id': 'UPS', 'tweets': 16}
{'_id': 'LOW', 'tweets': 14}
{'_id': 'BRK.B', 'tweets': 14}
{'_id': 'KO', 'tweets': 13}
{'_id': 'BKNG', 'tweets': 13}
{'_id': 'TSM', 'tweets': 10}
{'_id': 'MA', 'tweets': 8}
{'_id': 'UNH', 'twee

In [37]:
# Top 5 most tweeted tickers
query = [
    {"$group": {"_id": "$ticker", "total": {"$sum": 1}}},
    {"$sort": {"total": -1}},
    {"$limit": 5}
]

records = collection.aggregate(query)

for doc in records:
    print(doc)


{'_id': 'TSLA', 'total': 4341}
{'_id': 'AAPL', 'total': 1721}
{'_id': 'BA', 'total': 919}
{'_id': 'DIS', 'total': 432}
{'_id': 'AMZN', 'total': 407}


#### Create a new collection with the results from filtering the collection stocktweet with ticker = AAPL and data = 22/12/2020

In [35]:
# Filter with multiple conditions
collection_name_results = "stocktweetresults"

# Create or access the collection
collection_results = db[collection_name_results]

records = list(collection.find({
    "ticker": "AAPL",
    "date": "22/12/2020"
}))
for doc in records:
    print(doc)
    
# Insert records into the collection
try:

    collection_results.insert_many(records)
    
except Exception as e:
    print(f"An error occurred: {e}")    
    

{'_id': ObjectId('692995114988782702a92056'), 'id': 107793, 'date': '22/12/2020', 'ticker': 'AAPL', 'tweet': '$AAPL my calls expire in May $125 I aint selling😤😤😤 MAXIMIZE PROFITS BABY'}
{'_id': ObjectId('692995114988782702a9205c'), 'id': 107799, 'date': '22/12/2020', 'ticker': 'AAPL', 'tweet': '$AAPL Going in right direction after breakout 🚀🚀🚀'}
{'_id': ObjectId('692995114988782702a92062'), 'id': 107805, 'date': '22/12/2020', 'ticker': 'AAPL', 'tweet': '$AAPL This is one of the most shorted stocks on the market. Rocket fuel just needs a little spark!🚀🚀🚀'}
{'_id': ObjectId('692995114988782702a92064'), 'id': 107807, 'date': '22/12/2020', 'ticker': 'AAPL', 'tweet': '$AAPL 129$ call. Bought yesterday literally 1 minute before close. Should I sell today or hold on? 📈🚀🤷🏼\u200d♂️'}
{'_id': ObjectId('692995114988782702a92065'), 'id': 107808, 'date': '22/12/2020', 'ticker': 'AAPL', 'tweet': '$AAPL hey can y’all sell your shares so I can get more shares in at a better position that ja 😉😉'}
{'_id